This phase resolves the configured pair set and acquires every PMID and title returned by each pathogen–host query. PubMed queries include accepted pathogen names, mapped nomenclatural aliases, and taxon-checked ICTV virus names and abbreviations that uniquely map to one pathogen. Raw isolate/strain labels, unresolved names, and cross-taxon homonyms remain in taxonomy provenance but are excluded from search. The pair-search cache is the source corpus for the host-specific pathogen yes/no screen in Step 4. The configurable sample cap controls that screen; the pair searches themselves retain all results.

**Saved outputs are historical.** They were produced before host-mention expansion was added; rerun notebook 00, then this notebook, to refresh the query previews and acquisition results. 

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'pathogen_atlas_v4').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'pathogen_atlas_v4').is_dir():
    raise RuntimeError('Open this notebook from the V4 repository')
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / 'notebooks'))
from _shared import configure_notebook, configured_run_id, log_step, ncbi_contact_email, staging_root_for_config
RUN_ID = configured_run_id(PROJECT_ROOT)
ROOT, LOG, LOG_PATH = configure_notebook('03_acquire_pair_pubmed_records', RUN_ID)
print('Project root:', ROOT, '| log:', LOG_PATH)

2026-11-03 15:58 INFO Notebook session started at 2026-11-03T15:58:14.946356+00:00
2026-11-03 15:58 INFO Project root: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4
2026-11-03 15:58 INFO Run log: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/logs/human-parasites-2026-11-03/03_acquire_pair_pubmed_records.log


Project root: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4 | log: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/logs/human-parasites-2026-11-03/03_acquire_pair_pubmed_records.log


## Step 1 · Load configuration and taxonomy references

In [2]:
import yaml, pandas as pd
from IPython.display import display
from pathogen_atlas_v4.runs import resolve_run, save_resolved_run
from pathogen_atlas_v4.pubmed import build_pair_query
from pathogen_atlas_v4.pubmed_client import pubmed_aliases_by_pathogen
CONFIG = yaml.safe_load((ROOT / 'configs/pilot_v1.yaml').read_text())
HOST_SEARCH_EXPANSIONS = CONFIG['acquisition'].get('host_search_expansions', {})
HOST_EXPANSION_HASH = CONFIG['acquisition'].get('host_expansion_hash', '')
if CONFIG['acquisition'].get('host_expansion_enabled', False):
    configured_host_ids = set(map(str, CONFIG['run'].get('host_ids', [])))
    missing_expansion_hosts = sorted(configured_host_ids - set(HOST_SEARCH_EXPANSIONS))
    if missing_expansion_hosts:
        raise ValueError(f'Host expansion is enabled but terms are missing for {missing_expansion_hosts}; rerun notebook 00_configure_run.ipynb first.')
TAXONOMY_ROOT = staging_root_for_config(ROOT, CONFIG) / 'taxonomy'
RUN_TAXONOMY_ROOT = ROOT / 'data/v4/runs' / CONFIG['run']['run_id'] / 'taxonomy'
if not (RUN_TAXONOMY_ROOT / 'pathogens.parquet').exists():
    raise FileNotFoundError('Run taxonomy was not resolved; run 00_configure_run.ipynb first.')
hosts = pd.read_parquet(RUN_TAXONOMY_ROOT / 'hosts.parquet')
pathogens = pd.read_parquet(RUN_TAXONOMY_ROOT / 'pathogens.parquet')
ALIASES = pd.read_parquet(RUN_TAXONOMY_ROOT / 'pathogen_aliases.parquet')
ALIASES_BY_PATHOGEN, ALIAS_FILTER_COUNTS = pubmed_aliases_by_pathogen(pathogens, ALIASES)
LOG.info('PubMed alias safeguard: %s', ALIAS_FILTER_COUNTS)
LOG.info('Inputs loaded: hosts=%d pathogens=%d config=%s', len(hosts), len(pathogens), CONFIG['run']['run_id'])
display(CONFIG['run'])

2026-11-03 15:58 INFO PubMed alias safeguard: {'aliases_total': 332312, 'aliases_missing_name': 0, 'aliases_unmapped': 21291, 'aliases_non_nomenclatural': 90177, 'aliases_shared_across_pathogens': 3953, 'aliases_vmr_records_checked': 36743, 'aliases_vmr_terms_extracted': 65410, 'aliases_searchable': 36728}
2026-11-03 15:58 INFO Inputs loaded: hosts=11 pathogens=197775 config=human-parasites-2026-11-03


{'run_id': 'human-parasites-2026-11-03',
 'config_version': 'pilot-config-v2',
 'selection_mode': 'full',
 'selection_seed': 'human-parasites-2026-11-03',
 'host_ids': ['ncbi:9606'],
 'pathogen_ids': [],
 'pair_generation': 'cartesian_product',
 'pathogen_groups': ['parasites']}

## Step 2 · Resolve the exact pair set

In [3]:
host_groups = {key: group.host_id.astype(str).tolist() for key, group in hosts.groupby('host_group')}
pathogen_groups = {key: group.pathogen_id.astype(str).tolist() for key, group in pathogens.groupby('pathogen_group')}
resolved = resolve_run(CONFIG, known_host_ids=set(hosts.host_id.astype(str)),
                       known_pathogen_ids=set(pathogens.pathogen_id.astype(str)),
                       host_group_ids=host_groups, pathogen_group_ids=pathogen_groups)
RESOLVED_PATH = ROOT / 'data/v4/runs' / resolved.run_id / 'resolved_config.json'
save_resolved_run(CONFIG, resolved, RESOLVED_PATH)
print('Run:', resolved.run_id, '| generation rule:', resolved.pair_generation)
print('Hosts:', len(resolved.host_ids), '| pathogens:', len(resolved.pathogen_ids), '| pairs:', len(resolved.pairs))
print('Resolved configuration:', RESOLVED_PATH)
LOG.info('Resolved and saved run: hosts=%d pathogens=%d pairs=%d hash=%s', len(resolved.host_ids), len(resolved.pathogen_ids), len(resolved.pairs), resolved.resolved_config_hash)

2026-11-03 15:58 INFO Resolved and saved run: hosts=1 pathogens=57151 pairs=57151 hash=2d7fc8b91e2e2b58451fa3da7541d8ef70cac034be0f082fec7f8804726ad6b5


Run: human-parasites-2026-11-03 | generation rule: cartesian_product
Hosts: 1 | pathogens: 57151 | pairs: 57151
Resolved configuration: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/runs/human-parasites-2026-11-03/resolved_config.json


## Step 3 · Build and inspect pair-aware query text

In [4]:
host_rows = hosts.set_index('host_id')
pathogen_rows = pathogens.set_index('pathogen_id')
PREVIEW_ROWS = []
for pathogen_id, host_id in resolved.pairs[:10]:
    pathogen = pathogen_rows.loc[pathogen_id]
    host = host_rows.loc[host_id]
    host_aliases = host.get('aliases', [])
    if hasattr(host_aliases, 'tolist'):
        host_aliases = host_aliases.tolist()
    expansion = HOST_SEARCH_EXPANSIONS.get(str(host_id), {})
    host_mention_terms = [item['term'] for item in expansion.get('terms', [])]
    query = build_pair_query([pathogen.accepted_name, *ALIASES_BY_PATHOGEN.get(str(pathogen_id), [])],
                             [host.accepted_name, *(host_aliases or [])],
                             host_mention_terms=host_mention_terms, host_mesh_terms=expansion.get('mesh_terms', []),
                             strategy_version=CONFIG['acquisition']['query_strategy_version'])
    PREVIEW_ROWS.append({'pathogen_id': pathogen_id, 'pathogen': pathogen.accepted_name,
                         'host_id': host_id, 'host': host.accepted_name,
                         'host_cue_count': len(host_mention_terms), 'host_mesh_terms': expansion.get('mesh_terms', []), **query})
display(pd.DataFrame(PREVIEW_ROWS))
LOG.info('Generated %d preview queries; full acquisition covers all %d pairs with strategy %s', len(PREVIEW_ROWS), len(resolved.pairs), CONFIG['acquisition']['query_strategy_version'])


,pathogen_id,pathogen,host_id,host,host_cue_count,host_mesh_terms,query,query_hash,query_version
0,ncbi:2017973,Abbreviata antarctica,ncbi:9606,Homo sapiens,35,[Humans],"(""Abbreviata antarctica""[Title/Abstract]) AND ...",d054e4ee8177cee1bfcf3fe2e054861aa82febdf7a335b...,pathogen-host-expanded-host-mentions-mesh-titl...
1,ncbi:2699125,Abbreviata caucasica,ncbi:9606,Homo sapiens,35,[Humans],"(""Abbreviata caucasica""[Title/Abstract] OR ""Ph...",046836b0a211047893e7423185a8cbd45438c2259f8c5b...,pathogen-host-expanded-host-mentions-mesh-titl...
2,ncbi:2017974,Abbreviata hastaspicula,ncbi:9606,Homo sapiens,35,[Humans],"(""Abbreviata hastaspicula""[Title/Abstract]) AN...",7093071abf73e250e08b5681e809ad2aa82ec2fc121d6d...,pathogen-host-expanded-host-mentions-mesh-titl...
3,ncbi:2609951,Abbreviata kazachstanica,ncbi:9606,Homo sapiens,35,[Humans],"(""Abbreviata kazachstanica""[Title/Abstract]) A...",d4c28d7e403de4f6cfb9275037256eeb3678962fca397c...,pathogen-host-expanded-host-mentions-mesh-titl...
4,ncbi:3415043,Abbreviata sp.,ncbi:9606,Homo sapiens,35,[Humans],"(""Abbreviata sp.""[Title/Abstract]) AND (""adole...",2162a96725815159b83cb0ecc3cb45e6cd3e0ae7c850a9...,pathogen-host-expanded-host-mentions-mesh-titl...
5,ncbi:3443757,Abbreviata varani,ncbi:9606,Homo sapiens,35,[Humans],"(""Abbreviata varani""[Title/Abstract]) AND (""ad...",67253f525b1b93769ed84f64df72b2ca60c60eaf5d7eda...,pathogen-host-expanded-host-mentions-mesh-titl...
6,ncbi:1820257,Aberrapex sp. n. 1 AW-2016,ncbi:9606,Homo sapiens,35,[Humans],"(""Aberrapex sp. n. 1 AW-2016""[Title/Abstract])...",c88d8f8fa72e7f5fbed4e24c7e8f9eeb0409309cb2a965...,pathogen-host-expanded-host-mentions-mesh-titl...
7,ncbi:2977803,Aborjinia sp. 1 AAS-2022,ncbi:9606,Homo sapiens,35,[Humans],"(""Aborjinia sp. 1 AAS-2022""[Title/Abstract]) A...",2d8359afd512b21dc9849649628fddd5ce6c440e2c23ec...,pathogen-host-expanded-host-mentions-mesh-titl...
8,ncbi:2855635,Aborjinia sp. OVP-2021,ncbi:9606,Homo sapiens,35,[Humans],"(""Aborjinia sp. OVP-2021""[Title/Abstract]) AND...",954909f1c889a3462405a9c770bed9f39a6ad58e46e064...,pathogen-host-expanded-host-mentions-mesh-titl...
9,ncbi:70589,Abothrium gadi,ncbi:9606,Homo sapiens,35,[Humans],"(""Abothrium gadi""[Title/Abstract]) AND (""adole...",0b9c72971c17dcc5a0eb4d79730f1044afd84adb74c986...,pathogen-host-expanded-host-mentions-mesh-titl...


2026-11-03 15:58 INFO Generated 10 preview queries; full acquisition covers all 57151 pairs with strategy pathogen-host-expanded-host-mentions-mesh-title-abstract-v3


## Step 4 · Preflight counts and acquisition readiness

In [5]:
from pathogen_atlas_v4.pubmed_client import literature_cache_key
PAIR_COUNT = len(resolved.pairs)
UNIQUE_PATHOGENS = len(resolved.pathogen_ids)
UNIQUE_HOSTS = len(resolved.host_ids)
SAMPLING = CONFIG['acquisition'].get('sampling', {})
SAMPLE_MAXIMUM = SAMPLING.get('max_pmids_per_pair', 200)
if not isinstance(SAMPLE_MAXIMUM, int) or not 1 <= SAMPLE_MAXIMUM <= 200:
    raise ValueError('Full workflow screens at most 200 abstracts per pathogen–host pair; set max_pmids_per_pair to an integer from 1 to 200.')
SAMPLE_OLDEST = int(SAMPLING.get('oldest_pmids_per_pair', 30))
SAMPLE_SEED = int(SAMPLING.get('random_seed', 42))
SAMPLE_POLICY_VERSION = str(SAMPLING.get('sample_policy_version', 'stable-prefix-hash-v1'))
# Cache identity excludes the sample cap, host selector, and pathogen selector so overlapping acquisitions are reusable.
cache_key = literature_cache_key(query_version=CONFIG['acquisition']['query_strategy_version'],
                                  pathogen_literature_query_version=CONFIG['acquisition']['pathogen_literature_query_version'],
                                  taxonomy_snapshot=TAXONOMY_ROOT.parent.name,
                                  host_expansion_hash=HOST_EXPANSION_HASH)
CACHE_DIR = ROOT / 'data/v4/cache/literature' / cache_key
CACHE_DIR.mkdir(parents=True, exist_ok=True)
preflight = {'pairs_to_search': PAIR_COUNT, 'distinct_pathogens': UNIQUE_PATHOGENS, 'distinct_hosts': UNIQUE_HOSTS,
             'query_strategy_version': CONFIG['acquisition']['query_strategy_version'],
             'pair_checkpoint': str(CACHE_DIR / 'pair_search_checkpoints.jsonl'),
             'per_pair_queries': True, 'host_expansion_hash': HOST_EXPANSION_HASH,
             'host_expansion_model': CONFIG['acquisition'].get('host_expansion_model'),
             'screen_sample_cap_per_pair': SAMPLE_MAXIMUM,
             'oldest_pmids_per_pair': SAMPLE_OLDEST, 'random_seed': SAMPLE_SEED, 'sample_policy_version': SAMPLE_POLICY_VERSION, 'network_calls_made': 0}
preflight['pubmed_alias_filter'] = ALIAS_FILTER_COUNTS
display(preflight)
LOG.info('PubMed preflight: pairs=%d pathogens=%d hosts=%d cache=%s aliases=%d searchable_aliases=%d excluded_shared=%d excluded_non_nomenclatural=%d', PAIR_COUNT, UNIQUE_PATHOGENS, UNIQUE_HOSTS, CACHE_DIR, ALIAS_FILTER_COUNTS['aliases_total'], ALIAS_FILTER_COUNTS['aliases_searchable'], ALIAS_FILTER_COUNTS['aliases_shared_across_pathogens'], ALIAS_FILTER_COUNTS['aliases_non_nomenclatural'])


{'pairs_to_search': 57151,
 'distinct_pathogens': 57151,
 'distinct_hosts': 1,
 'query_strategy_version': 'pathogen-host-expanded-host-mentions-mesh-title-abstract-v3',
 'pair_checkpoint': '/Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/cache/literature/4b73154df150c014ba44424af6dc1ee25c67ac38132cc5f7caaa74728d87a961/pair_search_checkpoints.jsonl',
 'per_pair_queries': True,
 'host_expansion_hash': '56a08772b5996a94f955fd767fe89145b8d73931c8ad3d5d73c78155a929aaaf',
 'host_expansion_model': 'gpt-6.1-terra',
 'screen_sample_cap_per_pair': 200,
 'oldest_pmids_per_pair': 30,
 'random_seed': 42,
 'sample_policy_version': 'stable-prefix-hash-v1',
 'network_calls_made': 0,
 'pubmed_alias_filter': {'aliases_total': 332312,
  'aliases_missing_name': 0,
  'aliases_unmapped': 21291,
  'aliases_non_nomenclatural': 90177,
  'aliases_shared_across_pathogens': 3953,
  'aliases_vmr_records_checked': 36743,
  'aliases

2026-11-03 15:58 INFO PubMed preflight: pairs=57151 pathogens=57151 hosts=1 cache=/Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/cache/literature/4b73154df150c014ba44424af6dc1ee25c67ac38132cc5f7caaa74728d87a961 aliases=332312 searchable_aliases=36728 excluded_shared=3953 excluded_non_nomenclatural=90177


## Step 5 · Search every selected pair

The client searches every selected pathogen–host pair, retains the complete PMID set, and checkpoints each result for resumability. Requests run concurrently under a shared NCBI rate limiter. Article titles and abstracts are fetched later for the stable sample selected by the run configuration. Search result sets above PubMed’s 10,000-record UID retrieval ceiling are segmented by publication date; a segment that still cannot be retrieved completely raises an error instead of silently truncating.

In [6]:
import os
from pathogen_atlas_v4.pubmed_client import acquire_pubmed_pairs, ncbi_api_key

NCBI_EMAIL = ncbi_contact_email(ROOT)
NCBI_API_KEY = ncbi_api_key()
LOG.info('Starting full PubMed acquisition: pairs=%d api_key_configured=%s cache=%s', len(resolved.pairs), bool(NCBI_API_KEY), CACHE_DIR)
acquisition = acquire_pubmed_pairs(
    resolved.pairs, pathogens, hosts, ALIASES, CACHE_DIR,
    query_version=CONFIG['acquisition']['query_strategy_version'],
    host_search_expansions=HOST_SEARCH_EXPANSIONS,
    email=NCBI_EMAIL, api_key=NCBI_API_KEY, logger=LOG,
    sample_maximum=SAMPLE_MAXIMUM, sample_oldest_count=SAMPLE_OLDEST, sample_seed=SAMPLE_SEED,
    max_workers=CONFIG['acquisition'].get('max_concurrent_requests', 8),
)
display(acquisition)
LOG.info('Pair PubMed acquisition complete: %s', acquisition)
LOG.info('Pathogen-only total-publication counts are deferred until pathogen–host screening identifies the accepted pathogen set.')


2026-11-03 15:58 INFO Starting full PubMed acquisition: pairs=57151 api_key_configured=True cache=/Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/cache/literature/4b73154df150c014ba44424af6dc1ee25c67ac38132cc5f7caaa74728d87a961


Using the configured Git email as the NCBI contact address.


2026-11-03 15:58 INFO PubMed pair-search alias safeguard: {"aliases_missing_name": 0, "aliases_non_nomenclatural": 90177, "aliases_searchable": 36728, "aliases_shared_across_pathogens": 3953, "aliases_total": 332312, "aliases_unmapped": 21291, "aliases_vmr_records_checked": 36743, "aliases_vmr_terms_extracted": 65410}
2026-11-03 15:58 INFO PubMed pair acquisition settings: pairs=57151 already_checkpointed=49105 searches_to_run=8046; sample cap=200 oldest=30 seed=42; workers=8; API key=True; cached article PMIDs=0
2026-11-03 15:58 WARNING ESearch retry 1/6 after HTTPError; sleeping 1s
2026-11-03 15:58 INFO PubMed pair checkpoints 49205/57151 (86.1%); updated this session=100; ETA 20m
2026-11-03 15:58 INFO PubMed pair checkpoints 49305/57151 (86.3%); updated this session=200; ETA 18m
2026-11-03 15:58 WARNING ESearch retry 1/6 after HTTPError; sleeping 1s
2026-11-03 15:58 WARNING ESearch retry 1/6 after HTTPError; sleeping 1s
2026-11-03 15:58 WARNING ESearch retry 1/6 after HTTPError; sle

{'pairs_total': 57151,
 'pairs_complete': 57151,
 'pairs_updated_this_session': 8046,
 'pair_matches': 126223,
 'sample_selected_matches': 40028,
 'sample_cap_per_pair': 200,
 'oldest_per_pair': 30,
 'sample_seed': 42,
 'unique_sample_articles_cached': 0,
 'output_dir': '/Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/cache/literature/4b73154df150c014ba44424af6dc1ee25c67ac38132cc5f7caaa74728d87a961/tables',
 'checkpoint': '/Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/cache/literature/4b73154df150c014ba44424af6dc1ee25c67ac38132cc5f7caaa74728d87a961/pair_search_checkpoints.jsonl'}

2026-11-03 16:15 INFO Pair PubMed acquisition complete: {'pairs_total': 57151, 'pairs_complete': 57151, 'pairs_updated_this_session': 8046, 'pair_matches': 126223, 'sample_selected_matches': 40028, 'sample_cap_per_pair': 200, 'oldest_per_pair': 30, 'sample_seed': 42, 'unique_sample_articles_cached': 0, 'output_dir': '/Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/cache/literature/4b73154df150c014ba44424af6dc1ee25c67ac38132cc5f7caaa74728d87a961/tables', 'checkpoint': '/Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/cache/literature/4b73154df150c014ba44424af6dc1ee25c67ac38132cc5f7caaa74728d87a961/pair_search_checkpoints.jsonl'}
2026-11-03 16:15 INFO Pathogen-only total-publication counts are deferred until pathogen–host screening identifies the accepted pathogen set.
